# PMO Benchmark Comparison

This notebook compares our LLM-based molecule optimization results against the PMO benchmark baselines.

## Data Sources
- **Our results**: `data/results/tdc_tasks/` (JSON conversation files)
- **Baseline results**: `data/results/pmo_baseline/` (YAML files from PMO benchmark)

## Metrics
We use AUC top-10 as the primary metric, following the PMO benchmark paper.

In [1]:
import json
import heapq
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
# Define paths
OUR_RESULTS_DIR = Path("../data/results/tdc_tasks")
PMO_RESULTS_DIR = Path("../data/results/pmo_baseline")

# Define model mappings from filename to display name (PMO baselines)
MODEL_MAPPING = {
    'reinvent': 'REINVENT',
    'reinvent_selfies': 'REINVENT SELFIES',
    'graph_ga': 'Graph GA',
    'gp_bo': 'GP BO',
    'stoned': 'STONED',
    'smiles_lstm_hc': 'SMILES LSTM HC',
    'selfies_lstm_hc': 'SELFIES LSTM HC',
    'smiles_ga': 'SMILES GA',
    'selfies_ga': 'SELFIES GA',
    'smiles_vae_bo': 'SMILES VAE BO',
    'selfies_vae_bo': 'SELFIES VAE BO',
    'jt_vae_bo': 'JT-VAE BO',
    'synnet': 'SynNet',
    'dog_gen': 'DoG-Gen',
    'dog_ae': 'DoG-AE',
    'gflownet': 'GFlowNet',
    'gflownet_al': 'GFlowNet-AL',
    'dst': 'DST',
    'mars': 'MARS',
    'mimosa': 'MIMOSA',
    'pasithea': 'Pasithea',
    'mol_pal': 'MolPAL',
    'graph_mcts': 'Graph MCTS',
    'moldqn': 'MolDQN',
    'screening': 'Screening',
}

# Our model name
OUR_MODEL_KEY = 'llm_agent'
OUR_MODEL_NAME = 'LLM Agent (Ours)'

# List of all 23 PMO benchmark tasks
TASKS = [
    'albuterol_similarity',
    'amlodipine_mpo',
    'celecoxib_rediscovery',
    'deco_hop',
    'drd2',
    'fexofenadine_mpo',
    'gsk3b',
    'isomers_c7h8n2o2',
    'isomers_c9h10n2o2pf2cl',
    'jnk3',
    'median1',
    'median2',
    'mestranol_similarity',
    'osimertinib_mpo',
    'perindopril_mpo',
    'qed',
    'ranolazine_mpo',
    'scaffold_hop',
    'sitagliptin_mpo',
    'thiothixene_rediscovery',
    'troglitazone_rediscovery',
    'valsartan_smarts',
    'zaleplon_mpo',
]

## Load Our Results

In [3]:
def load_trace(p: Path) -> list:
    """Load trace from JSON file.
    
    The trace contains a list of entries with 'iteration', 'score', and 'smiles' fields.
    """
    d = json.loads(p.read_text(encoding="utf-8"))
    return d["trace"] if isinstance(d, dict) and "trace" in d else d


def load_our_results(results_dir: Path) -> dict:
    """Load all our results from JSON trace files.
    
    Returns a dict mapping task -> list of (score, iteration) tuples for each run.
    """
    all_results = defaultdict(list)
    
    for task_dir in results_dir.iterdir():
        if not task_dir.is_dir():
            continue
        
        task_name = task_dir.name
        
        for json_file in task_dir.glob("*.json"):
            try:
                trace = load_trace(json_file)
                # Convert trace entries to (score, iteration) tuples
                results = [(float(entry["score"]), int(entry["iteration"])) for entry in trace]
                # Sort by iteration
                results.sort(key=lambda x: x[1])
                
                if results:
                    all_results[task_name].append(results)
            except Exception as e:
                print(f"Error loading {json_file}: {e}")
    
    return dict(all_results)


# Load our results
our_results = load_our_results(OUR_RESULTS_DIR)
print(f"Loaded results for {len(our_results)} tasks")
print(f"Tasks: {list(our_results.keys())}")

Loaded results for 23 tasks
Tasks: ['deco_hop', 'sitagliptin_mpo', 'troglitazone_rediscovery', 'isomers_c11h24', 'median1', 'thiothixene_rediscovery', 'valsartan_smarts', 'amlodipine_mpo', 'drd2', 'osimertinib_mpo', 'mestranol_similarity', 'celecoxib_rediscovery', 'scaffold_hop', 'albuterol_similarity', 'jnk3', 'fexofenadine_mpo', 'perindopril_mpo', 'median2', 'ranolazine_mpo', 'isomers_c9h10n2o2pf2cl', 'qed', 'gsk3b', 'zaleplon_mpo']


## Load PMO Baseline Results

In [ ]:
def load_yaml_results_fast(filepath):
    """Load results from a YAML file using fast custom parsing.
    
    The YAML file format is:
    SMILES:
    - score
    - oracle_call_number
    
    Returns a list of (score, oracle_call) tuples sorted by oracle_call.
    """
    results = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    i = 0
    n = len(lines)
    while i < n:
        line = lines[i]
        if not line.strip():
            i += 1
            continue
        if not line.startswith('-'):
            if i + 2 < n:
                score_line = lines[i + 1].strip()
                oracle_line = lines[i + 2].strip()
                if score_line.startswith('- ') and oracle_line.startswith('- '):
                    try:
                        score = float(score_line[2:])
                        oracle_call = int(oracle_line[2:])
                        results.append((score, oracle_call))
                    except (ValueError, IndexError):
                        pass
                    i += 3
                    continue
        i += 1
    
    results.sort(key=lambda x: x[1])
    return results


# We need to load raw results and recompute AUC with k=1
# The existing cache uses k=10, so we compute fresh from YAML files

def load_pmo_raw_results():
    """Load raw PMO results from YAML files.
    
    Returns dict: model -> task -> list of [(score, oracle_call), ...] for each run
    """
    all_results = defaultdict(lambda: defaultdict(list))
    
    for model in MODEL_MAPPING.keys():
        for task in TASKS:
            pattern = f"results_{model}_{task}_*.yaml"
            for filepath in PMO_RESULTS_DIR.glob(pattern):
                results = load_yaml_results_fast(filepath)
                if results:
                    all_results[model][task].append(results)
    
    return all_results


print("Loading PMO baseline raw results from YAML files...")
pmo_raw_results = load_pmo_raw_results()
n_files = sum(len(runs) for model_data in pmo_raw_results.values() for runs in model_data.values())
print(f"Loaded {n_files} result files for {len(pmo_raw_results)} models")

Loading PMO baseline raw results from YAML files...


## Compute AUC Top-10 for Our Results

In [ ]:
MAX_ORACLE_CALLS = 10000  # Standard PMO benchmark limit


def compute_auc_top_1(results, max_oracle_calls=10000):
    """Compute the AUC of top-1 (best) score vs oracle calls.
    
    Args:
        results: List of (score, oracle_call) tuples sorted by oracle_call
        max_oracle_calls: Maximum number of oracle calls (default 10000)
    
    Returns:
        AUC value normalized to [0, 1]
    """
    if not results:
        return 0.0
    
    best_score = 0.0
    best_at_call = {}
    
    for score, oracle_call in results:
        if oracle_call > max_oracle_calls:
            break
        
        if score > best_score:
            best_score = score
        
        best_at_call[oracle_call] = best_score
    
    if not best_at_call:
        return 0.0
    
    oracle_calls = sorted(best_at_call.keys())
    
    # Compute AUC as sum of rectangles
    auc = 0.0
    prev_call = 0
    prev_best = 0.0
    
    for call in oracle_calls:
        auc += prev_best * (call - prev_call)
        prev_call = call
        prev_best = best_at_call[call]
    
    # Add final rectangle from last call to max_oracle_calls
    # This extends the best value to max_oracle_calls
    auc += prev_best * (max_oracle_calls - prev_call)
    
    # Normalize by max possible AUC
    normalized_auc = auc / max_oracle_calls
    
    return normalized_auc


# Compute AUC Top-1 for our results
# Our experiments use 51 iterations, but we extend to 10000 by holding best value constant
our_auc_results = defaultdict(list)

for task, runs in our_results.items():
    for run_results in runs:
        # Compute AUC over 10000 oracle calls
        # The best value at iteration 51 will be held constant until 10000
        auc = compute_auc_top_1(run_results, max_oracle_calls=MAX_ORACLE_CALLS)
        our_auc_results[task].append(auc)

print("AUC Top-1 for our results (extended to 10000 oracle calls):")
for task, aucs in sorted(our_auc_results.items()):
    print(f"  {task}: {np.mean(aucs):.4f} (n={len(aucs)})")

In [ ]:
# Compute AUC Top-1 for PMO baselines
pmo_auc_results = defaultdict(lambda: defaultdict(list))

for model, task_data in pmo_raw_results.items():
    for task, runs in task_data.items():
        for run_results in runs:
            auc = compute_auc_top_1(run_results, max_oracle_calls=MAX_ORACLE_CALLS)
            pmo_auc_results[model][task].append(auc)

print(f"Computed AUC Top-1 for {len(pmo_auc_results)} PMO baseline models")

## Create Comparison Table

In [ ]:
def create_comparison_table(pmo_results_dict, our_results_dict, models_order):
    """Create a comparison table with mean ± std for each model/task combination."""
    rows = []
    
    for task in TASKS:
        row = {'Task': task}
        
        # Add our results first
        if task in our_results_dict:
            values = our_results_dict[task]
            mean = np.mean(values)
            if len(values) > 1:
                std = np.std(values)
                row[OUR_MODEL_NAME] = f"{mean:.3f}± {std:.3f}"
            else:
                row[OUR_MODEL_NAME] = f"{mean:.3f}"
        else:
            row[OUR_MODEL_NAME] = "-"
        
        # Add PMO baselines
        for model in models_order:
            if model in pmo_results_dict and task in pmo_results_dict[model]:
                values = pmo_results_dict[model][task]
                mean = np.mean(values)
                std = np.std(values)
                row[MODEL_MAPPING[model]] = f"{mean:.3f}± {std:.3f}"
            else:
                row[MODEL_MAPPING[model]] = "-"
        rows.append(row)
    
    # Add sum row
    sum_row = {'Task': 'Sum'}
    
    # Our sum
    our_total = 0
    for task in TASKS:
        if task in our_results_dict:
            our_total += np.mean(our_results_dict[task])
    sum_row[OUR_MODEL_NAME] = f"{our_total:.3f}"
    
    # PMO baselines sums
    for model in models_order:
        if model in pmo_results_dict:
            total = 0
            for task in TASKS:
                if task in pmo_results_dict[model]:
                    total += np.mean(pmo_results_dict[model][task])
            sum_row[MODEL_MAPPING[model]] = f"{total:.3f}"
        else:
            sum_row[MODEL_MAPPING[model]] = "-"
    rows.append(sum_row)
    
    df = pd.DataFrame(rows)
    return df

In [ ]:
# Calculate sum for each PMO model
all_pmo_models = list(pmo_auc_results.keys())
model_sums = {}
for model in all_pmo_models:
    total = sum(np.mean(pmo_auc_results[model][task]) for task in TASKS if task in pmo_auc_results[model])
    model_sums[model] = total

# Sort models by sum (descending)
sorted_pmo_models = sorted(all_pmo_models, key=lambda m: model_sums[m], reverse=True)

print("\n" + "="*100)
print("COMPARISON TABLE: LLM Agent vs PMO Baselines (AUC Top-1)")
print("="*100 + "\n")

comparison_table = create_comparison_table(pmo_auc_results, our_auc_results, sorted_pmo_models)

# Add rank row
# Calculate our rank based on sum
our_sum = sum(np.mean(our_auc_results[task]) for task in TASKS if task in our_auc_results)
all_sums = [(OUR_MODEL_KEY, our_sum)] + [(m, model_sums[m]) for m in sorted_pmo_models]
all_sums_sorted = sorted(all_sums, key=lambda x: x[1], reverse=True)
rank_map = {model: i+1 for i, (model, _) in enumerate(all_sums_sorted)}

rank_row = {'Task': 'Rank'}
rank_row[OUR_MODEL_NAME] = str(rank_map[OUR_MODEL_KEY])
for model in sorted_pmo_models:
    rank_row[MODEL_MAPPING[model]] = str(rank_map[model])

comparison_table = pd.concat([comparison_table, pd.DataFrame([rank_row])], ignore_index=True)

display(comparison_table)

## Summary Statistics

In [ ]:
# Print summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

# Tasks we have results for
our_tasks = set(our_auc_results.keys())
pmo_tasks = set(TASKS)
common_tasks = our_tasks & pmo_tasks
missing_tasks = pmo_tasks - our_tasks
extra_tasks = our_tasks - pmo_tasks

print(f"\nTasks in PMO benchmark: {len(pmo_tasks)}")
print(f"Tasks with our results: {len(our_tasks)}")
print(f"Common tasks: {len(common_tasks)}")

if missing_tasks:
    print(f"\nMissing tasks (in PMO but not in ours): {sorted(missing_tasks)}")
if extra_tasks:
    print(f"\nExtra tasks (in ours but not in PMO): {sorted(extra_tasks)}")

# Our rank
print(f"\n" + "-"*40)
print(f"Our rank: {rank_map[OUR_MODEL_KEY]} out of {len(all_sums)}")
print(f"Our sum: {our_sum:.3f}")
print(f"Top model sum: {all_sums_sorted[0][1]:.3f} ({MODEL_MAPPING.get(all_sums_sorted[0][0], all_sums_sorted[0][0])})")

In [ ]:
# Show top 10 models by sum
print("\nTop 10 models by sum:")
print("-" * 40)
for i, (model, total) in enumerate(all_sums_sorted[:10], 1):
    name = OUR_MODEL_NAME if model == OUR_MODEL_KEY else MODEL_MAPPING.get(model, model)
    marker = " <-- Ours" if model == OUR_MODEL_KEY else ""
    print(f"{i:2d}. {name:25s} {total:.3f}{marker}")